In [1]:
!pip install -q datasets transformers accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
import pandas as pd

In [3]:
import datasets
import transformers
import huggingface_hub

print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

datasets: 4.8.5
transformers: 5.16.1
huggingface_hub: 1.29.0


In [4]:
from datasets import load_dataset

dataset = load_dataset("rajpurkar/squad")

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [5]:
print("Training Samples :", len(dataset["train"]))
print("Validation Samples :", len(dataset["validation"]))

Training Samples : 87599
Validation Samples : 10570


In [6]:
dataset["train"].column_names

['id', 'title', 'context', 'question', 'answers']

In [7]:
sample = dataset["train"][0]

print("TITLE\n")
print(sample["title"])

print("\n" + "="*80)

print("\nCONTEXT\n")
print(sample["context"])

print("\n" + "="*80)

print("\nQUESTION\n")
print(sample["question"])

print("\n" + "="*80)

print("\nANSWER\n")
print(sample["answers"])

TITLE

University_of_Notre_Dame


CONTEXT

Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.


QUESTION

To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?


ANSWER

{'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}


In [8]:
import pandas as pd

train_df = dataset["train"].to_pandas()

In [9]:
train_df.head()

,id,title,context,question,answers
0,5733be284776f41900661182,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",To whom did the Virgin Mary allegedly appear i...,"{'text': ['Saint Bernadette Soubirous'], 'answ..."
1,5733be284776f4190066117f,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is in front of the Notre Dame Main Building?,"{'text': ['a copper statue of Christ'], 'answe..."
2,5733be284776f41900661180,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",The Basilica of the Sacred heart at Notre Dame...,"{'text': ['the Main Building'], 'answer_start'..."
3,5733be284776f41900661181,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What is the Grotto at Notre Dame?,{'text': ['a Marian place of prayer and reflec...
4,5733be284776f4190066117e,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha...",What sits on top of the Main Building at Notre...,{'text': ['a golden statue of the Virgin Mary'...


In [10]:
train_df.isnull().sum()

,0
id,0
title,0
context,0
question,0
answers,0


In [11]:
train_df.sample(5)

,id,title,context,question,answers
18123,56eaa5f15a205f1900d6d3c2,Political_corruption,Electoral fraud is illegal interference with t...,Another term for electoral fraud is what?,"{'text': ['voter fraud'], 'answer_start': [276]}"
26049,570714f39e06ca38007e93c4,Chihuahua_(state),Santa Bárbara became the launching place for e...,In which year was El Paso del Norte found?,"{'text': ['1598'], 'answer_start': [316]}"
19994,56f78c18a6d7ea1400e17257,Marshall_Islands,Spanish explorer Alonso de Salazar was the fir...,What was the name of Alonso de Salazar's ship?,"{'text': ['Santa Maria de la Victoria'], 'answ..."
39460,5727d0143acd2414000ded02,Northwestern_University,Under Walter Dill Scott's presidency from 1920...,What was Northwestern one of the first six uni...,{'text': ['a Naval Reserve Officers Training C...
78491,5732275d0fdd8d15006c67f8,Pacific_War,Although the advance in the Arakan had been ha...,What fortified position was captured by the Ch...,"{'text': ['Mount Song'], 'answer_start': [392]}"


In [12]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [13]:
from transformers import AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained(
    "distilbert-base-uncased"
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
qa_outputs.weight       | MISSING    | 
qa_outputs.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
max_length = 384
stride = 128

In [15]:
def preprocess_function(examples):

    questions = [q.strip() for q in examples["question"]]

    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):

        sample_idx = sample_map[i]

        answer = examples["answers"][sample_idx]

        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1

        context_start = idx

        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1

        context_end = idx - 1

        if offsets[context_start][0] > end_char or offsets[context_end][1] < start_char:

            start_positions.append(0)
            end_positions.append(0)

        else:

            idx = context_start

            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1

            start_positions.append(idx - 1)

            idx = context_end

            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1

            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    return inputs

In [16]:
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [17]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 88492
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 10753
    })
})

In [18]:
tokenized_dataset["train"][0]

{'input_ids': [101,
  2000,
  3183,
  2106,
  1996,
  6261,
  2984,
  9382,
  3711,
  1999,
  8517,
  1999,
  10223,
  26371,
  2605,
  1029,
  102,
  6549,
  2135,
  1010,
  1996,
  2082,
  2038,
  1037,
  3234,
  2839,
  1012,
  10234,
  1996,
  2364,
  2311,
  1005,
  1055,
  2751,
  8514,
  2003,
  1037,
  3585,
  6231,
  1997,
  1996,
  6261,
  2984,
  1012,
  3202,
  1999,
  2392,
  1997,
  1996,
  2364,
  2311,
  1998,
  5307,
  2009,
  1010,
  2003,
  1037,
  6967,
  6231,
  1997,
  4828,
  2007,
  2608,
  2039,
  14995,
  6924,
  2007,
  1996,
  5722,
  1000,
  2310,
  3490,
  2618,
  4748,
  2033,
  18168,
  5267,
  1000,
  1012,
  2279,
  2000,
  1996,
  2364,
  2311,
  2003,
  1996,
  13546,
  1997,
  1996,
  6730,
  2540,
  1012,
  3202,
  2369,
  1996,
  13546,
  2003,
  1996,
  24665,
  23052,
  1010,
  1037,
  14042,
  2173,
  1997,
  7083,
  1998,
  9185,
  1012,
  2009,
  2003,
  1037,
  15059,
  1997,
  1996,
  24665,
  23052,
  2012,
  10223,
  26371,
  1010,
  2605

In [19]:
print(tokenized_dataset["train"][0].keys())

dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'])


In [20]:
print("Start Position:", tokenized_dataset["train"][0]["start_positions"])
print("End Position:", tokenized_dataset["train"][0]["end_positions"])

Start Position: 130
End Position: 137


In [21]:
count = 0

for sample in tokenized_dataset["train"]:
    if sample["start_positions"] == 0 and sample["end_positions"] == 0:
        count += 1

print(f"Samples with (0,0): {count}")
print(f"Total samples: {len(tokenized_dataset['train'])}")

Samples with (0,0): 759
Total samples: 88492


In [22]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [23]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="./distilbert_squad",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_steps=500,

    save_total_limit=2,

    load_best_model_at_end=True,

    report_to="none",

    fp16=torch.cuda.is_available()
)

In [24]:
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator()

In [25]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.156297,1.128294
2,0.879305,1.116440


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=22124, training_loss=1.1715501875973076, metrics={'train_runtime': 2183.1339, 'train_samples_per_second': 81.069, 'train_steps_per_second': 10.134, 'total_flos': 1.7342631192047616e+16, 'train_loss': 1.1715501875973076, 'epoch': 2.0})

In [27]:
trainer.save_model("distilbert_squad_model")
tokenizer.save_pretrained("distilbert_squad_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('distilbert_squad_model/tokenizer_config.json',
 'distilbert_squad_model/tokenizer.json')

In [28]:
import transformers

print(transformers.__version__)

5.16.1


In [29]:
from transformers.pipelines import SUPPORTED_TASKS

print(SUPPORTED_TASKS.keys())

dict_keys(['audio-classification', 'automatic-speech-recognition', 'text-to-audio', 'feature-extraction', 'text-classification', 'token-classification', 'table-question-answering', 'document-question-answering', 'fill-mask', 'text-generation', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-audio-classification', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'object-detection', 'zero-shot-object-detection', 'depth-estimation', 'video-classification', 'mask-generation', 'keypoint-matching', 'any-to-any'])


In [30]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

tokenizer = AutoTokenizer.from_pretrained("./distilbert_squad_model")
model = AutoModelForQuestionAnswering.from_pretrained("./distilbert_squad_model")

context = """
Python is a high-level programming language created by Guido van Rossum.
It supports object-oriented programming.
"""

question = "Who created Python?"

inputs = tokenizer(
    question,
    context,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(**inputs)

start = torch.argmax(outputs.start_logits)
end = torch.argmax(outputs.end_logits) + 1

answer = tokenizer.decode(
    inputs["input_ids"][0][start:end],
    skip_special_tokens=True
)

print("Predicted Answer:", answer)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Predicted Answer: guido van rossum


In [31]:
import torch

def ask_question(context, question):
    inputs = tokenizer(question, context, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    start = torch.argmax(outputs.start_logits)
    end = torch.argmax(outputs.end_logits) + 1

    answer = tokenizer.decode(
        inputs["input_ids"][0][start:end],
        skip_special_tokens=True
    )

    print(f"Question : {question}")
    print(f"Answer   : {answer}")

In [32]:
context = """
Python is a high-level programming language created by Guido van Rossum.
"""

ask_question(context, "Who created Python?")

Question : Who created Python?
Answer   : guido van rossum


In [33]:
!pip install -q evaluate

In [34]:
import evaluate

metric = evaluate.load("squad")

In [35]:
import torch

predictions = []
references = []

validation_dataset = dataset["validation"].select(range(100))

In [36]:
for sample in validation_dataset:

    context = sample["context"]
    question = sample["question"]

    inputs = tokenizer(question, context, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    start = torch.argmax(outputs.start_logits)
    end = torch.argmax(outputs.end_logits) + 1

    answer = tokenizer.decode(
        inputs["input_ids"][0][start:end],
        skip_special_tokens=True
    )

    predictions.append({
        "id": sample["id"],
        "prediction_text": answer
    })

    references.append({
        "id": sample["id"],
        "answers": sample["answers"]
    })

In [37]:
results = metric.compute(
    predictions=predictions,
    references=references
)

print(results)

{'exact_match': 75.0, 'f1': 81.25531135531138}


In [38]:
validation_dataset = dataset["validation"]

In [39]:
print(len(validation_dataset))

10570


In [40]:
predictions = []
references = []

for sample in validation_dataset:

    context = sample["context"]
    question = sample["question"]

    inputs = tokenizer(
        question,
        context,
        max_length=384,
        truncation="only_second",
        padding="max_length",
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)

    start = torch.argmax(outputs.start_logits)
    end = torch.argmax(outputs.end_logits) + 1

    answer = tokenizer.decode(
        inputs["input_ids"][0][start:end],
        skip_special_tokens=True
    )

    predictions.append({
        "id": sample["id"],
        "prediction_text": answer
    })

    references.append({
        "id": sample["id"],
        "answers": sample["answers"]
    })

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.


KeyboardInterrupt



In [41]:
 results = metric.compute(
    predictions=predictions,
    references=references
)

print(results)

{'exact_match': 70.59961315280464, 'f1': 78.78677866023557}


In [42]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [43]:
!cp -r distilbert_squad_model /content/drive/MyDrive/

In [44]:
!zip -r distilbert_squad_model.zip distilbert_squad_model

  adding: distilbert_squad_model/ (stored 0%)
  adding: distilbert_squad_model/config.json (deflated 48%)
  adding: distilbert_squad_model/training_args.bin (deflated 53%)
  adding: distilbert_squad_model/tokenizer.json (deflated 71%)
  adding: distilbert_squad_model/model.safetensors (deflated 8%)
  adding: distilbert_squad_model/tokenizer_config.json (deflated 43%)


In [45]:
from google.colab import files

files.download("distilbert_squad_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>